# LLM capability-floor smoke (M4)

**What this does:** makes a small open model (Qwen2.5-3B) *play* the beer game, with no training. This is the free "can the model even play coherently?" check that PROJECT_SPEC calls for **before** any paid GRPO run.

**How to use:** Runtime -> Change runtime type -> **T4 GPU**. Then run the cells top to bottom (Runtime -> **Run all**). The first cell takes a few minutes (it installs the model server). Total cost: **$0** on Colab free tier.

**What to look for:** in the last cell's output, does the model keep inventory and cost in a sane range (not ordering 0 forever or slamming the cap), and is `parse_fail_rate` near 0? If yes, the capability floor passes and GRPO is worth doing. If the model plays incoherently, the spec says move up a model size before spending anything.

In [ ]:
#@title 1) Install Ollama + start the model server
# Ollama is a tiny program that runs an open LLM locally and answers requests.
# Our decoder talks to it over http://127.0.0.1:11434 (nothing to configure).
# The installer's archive is zstd-compressed, so we install zstd first
# (that was the earlier failure), then run the official script, with a
# direct GitHub download as a backup. Errors are printed in full.
import subprocess, time, os, urllib.request

def sh(cmd):
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout.strip():
        print(p.stdout[-1500:])
    if p.returncode != 0 and p.stderr.strip():
        print("STDERR:", p.stderr[-1500:])
    return p.returncode

def ollama_bin():
    for b in ("/usr/local/bin/ollama", "/usr/bin/ollama"):
        if os.path.exists(b):
            return b
    return None

# zstd is required to unpack the installer's archive (the earlier error).
sh("apt-get update -qq && apt-get install -y -qq zstd")

# Method A: official install script (ollama.com, not ollama.ai).
sh("curl -fsSL https://ollama.com/install.sh | sh")

# Method B fallback: download the release tarball from GitHub and unpack it.
# tar auto-detects the zstd compression now that zstd is installed.
if ollama_bin() is None:
    print("install script did not place the binary; trying direct download...")
    sh("curl -fL https://github.com/ollama/ollama/releases/latest/download/ollama-linux-amd64.tgz -o /tmp/ollama.tgz")
    sh("mkdir -p /usr/local && tar -C /usr/local -xf /tmp/ollama.tgz")

BIN = ollama_bin()
assert BIN, "Could not install Ollama — see output above."
print("ollama binary:", BIN)

# Start the server in the background.
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
subprocess.Popen([BIN, "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Wait until the API answers before continuing.
up = False
for _ in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2)
        up = True
        break
    except Exception:
        time.sleep(2)
assert up, "Ollama server did not come up — see output above."
print("ollama server is up")

In [ ]:
#@title 2) Download the model (Qwen2.5-3B)
MODEL = "qwen2.5:3b"  #@param {type:"string"}
import subprocess
subprocess.run(["ollama", "pull", MODEL], check=True)
print("pulled", MODEL)

In [ ]:
#@title 3) Clone the repo + install the environment
REPO = "ConstantinVictorBeatErtel/beer_distribution_RL"  #@param {type:"string"}
BRANCH = "claude/project-status-summary-zy3kht"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
# If the repo is PRIVATE, paste a GitHub token here (github.com -> Settings ->
# Developer settings -> Personal access tokens -> Fine-grained, read access to
# this repo). If the repo is PUBLIC, leave GITHUB_TOKEN blank.
import pathlib, subprocess, sys
auth = f"{GITHUB_TOKEN}@" if GITHUB_TOKEN else ""
REPO_URL = f"https://{auth}github.com/{REPO}.git"
repo = pathlib.Path("/content/beer_distribution_RL")

def run(cmd):
    p = subprocess.run(cmd, capture_output=True, text=True)
    if p.returncode != 0:
        err = p.stderr.strip()
        if GITHUB_TOKEN:
            err = err.replace(GITHUB_TOKEN, "***")
        print("STDERR:", err[-1500:])
        raise SystemExit(
            "git failed (see above). If it says 'repository not found' or asks "
            "for a username, the repo is PRIVATE: paste a token into GITHUB_TOKEN "
            "and run this cell again."
        )
    return p

if not repo.exists():
    run(["git", "clone", "--branch", BRANCH, "--depth", "1", REPO_URL, str(repo)])
else:
    run(["git", "-C", str(repo), "fetch", "--depth", "1", "origin", BRANCH])
    run(["git", "-C", str(repo), "checkout", BRANCH])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."], cwd=str(repo))
print("repo ready on branch", BRANCH)

In [ ]:
#@title 4a) Sanity check with NO model (demand-matching baseline)
# Proves the wiring works and gives the cost the LLM has to beat/match.
import subprocess, sys
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "heuristic", "--cell", "classic", "--horizon", "52",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 4b) The real capability floor: let Qwen play (classic cell)
# ~1 order per role per week x 52 weeks x 4 roles = ~200 model calls. A few minutes on T4.
import subprocess, sys
MODEL = "qwen2.5:3b"
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "ollama", "--model", MODEL, "--cell", "classic",
    "--horizon", "52", "--out", "/content/llm_floor_classic.json",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 4c) Capability floor on the first-GRPO-cell env (Y-topology, tight capacity)
import subprocess, sys
MODEL = "qwen2.5:3b"
subprocess.check_call([
    sys.executable, "scripts/run_llm_episode.py",
    "--backend", "ollama", "--model", MODEL, "--cell", "y_tight",
    "--horizon", "52", "--out", "/content/llm_floor_y_tight.json",
], cwd="/content/beer_distribution_RL")

In [ ]:
#@title 5) Verdict: did the model play coherently?
import json
for f in ["/content/llm_floor_classic.json", "/content/llm_floor_y_tight.json"]:
    d = json.load(open(f))
    print("===", d["cell"], "| model:", d["model"], "===")
    print("  parse_fail_rate:", d["parse_fail_rate"], "(want ~0)")
    print("  system_total_cost:", d["system_total_cost"])
    print("  mean order per role:", d["mean_order_per_role"])
    print("  first weeks:")
    for w in d["transcript_head"]:
        print("   ", w)
    print()